# RelCheck — Synthetic Hallucination Test

**Approach:** Instead of relying on VG to find naturally-occurring hallucinations,
we **inject** known hallucinations into accurate captions and measure whether
RelCheck detects and corrects them.

**Steps:**
1. Generate captions for R-Bench images (BLIP-2 / LLaVA / Qwen)
2. Extract relational triples from each caption
3. Inject hallucinations: swap relations with contradictory ones
4. Run RelCheck verifier on corrupted captions → measure **detection recall**
5. Run RelCheck corrector → measure **correction accuracy**
6. R-POPE LLM-judge on original vs corrupted vs corrected → measure **accuracy delta**

**Why this works:** We have perfect ground truth — we know exactly which triple
was corrupted and what the original relation was. No entity matching problem.


In [ ]:
# ============================================================
# Cell 1 — Setup
# ============================================================
!pip install together Pillow requests transformers -q

import os, json, re, time, random, requests, base64, zipfile, io, csv
from pathlib import Path
from collections import Counter, defaultdict
from io import BytesIO
from PIL import Image
from together import Together
from google.colab import drive

# ── Config ──────────────────────────────────────────────────
TOGETHER_API_KEY = ''   # <-- paste your Together.ai key
N_IMAGES         = 20   # start small for validation
CAPTIONER        = 'llava'   # 'blip2' | 'llava' | 'qwen'
RANDOM_SEED      = 42

# Drive paths (reuse RelCheck_Data from the 600-image run)
DRIVE_IMAGES_DIR = '/content/drive/MyDrive/RelCheck_Data/images'
RBENCH_PATH      = '/content/drive/MyDrive/RelCheck_Data/rbench_data.json'
SAVE_DIR         = '/content/drive/MyDrive/RelCheck_Data/synthetic_test'

LLM_MODEL    = 'meta-llama/Llama-3.3-70B-Instruct-Turbo'
VLM_MODEL    = 'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8'

_CAPTIONER_MODELS = {
    'blip2': None,   # loaded locally
    'qwen':  'Qwen/Qwen3-VL-8B-Instruct',
    'llava': 'llava-hf/llava-1.5-7b-hf',   # loaded locally
}

DETECTION_THRESHOLD = 0.25
GDINO_ID = 'IDEA-Research/grounding-dino-tiny'

drive.mount('/content/drive')
os.makedirs(SAVE_DIR, exist_ok=True)
os.environ['TOGETHER_API_KEY'] = TOGETHER_API_KEY
client = Together(api_key=TOGETHER_API_KEY)
random.seed(RANDOM_SEED)

def llm_call(messages, model=LLM_MODEL, max_tokens=600, retries=3):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages, temperature=0.0, max_tokens=max_tokens)
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retries - 1: time.sleep(2 ** attempt)
            else: print(f'  API failed: {e}'); return None

def vlm_call(messages, max_tokens=10, retries=3):
    return llm_call(messages, model=VLM_MODEL, max_tokens=max_tokens, retries=retries)

print('Setup done.')

# ── Load GDino ────────────────────────────────────────────
import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading GroundingDINO on {DEVICE}...')
gdino_processor = AutoProcessor.from_pretrained(GDINO_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(DEVICE)
print('  GroundingDINO ready.')

# ── Load LLaVA locally if needed ──────────────────────────
llava_model_local = llava_processor_local = None
if CAPTIONER == 'llava':
    try:
        from transformers import LlavaForConditionalGeneration, AutoProcessor as _AP
        _LLAVA_ID = 'llava-hf/llava-1.5-7b-hf'
        print(f'Loading LLaVA-1.5-7B locally on {DEVICE}...')
        llava_processor_local = _AP.from_pretrained(_LLAVA_ID)
        llava_model_local = LlavaForConditionalGeneration.from_pretrained(
            _LLAVA_ID, torch_dtype=torch.float16, device_map='auto')
        print('  LLaVA-1.5-7B ready.')
    except Exception as e:
        print(f'  LLaVA local load failed: {e}')

# ── Load BLIP-2 locally if needed ─────────────────────────
blip2_model_local = blip2_processor_local = None
if CAPTIONER == 'blip2':
    try:
        from transformers import Blip2ForConditionalGeneration, Blip2Processor
        _BLIP2_ID = 'Salesforce/blip2-flan-t5-xl'
        print(f'Loading BLIP-2 locally on {DEVICE}...')
        blip2_processor_local = Blip2Processor.from_pretrained(_BLIP2_ID)
        blip2_model_local = Blip2ForConditionalGeneration.from_pretrained(
            _BLIP2_ID, torch_dtype=torch.float16, device_map='auto')
        print('  BLIP-2 ready.')
    except Exception as e:
        print(f'  BLIP-2 local load failed: {e}')

# ── RelCheck verifier helpers (from VG notebook) ──────────
_SPATIAL_RELS = {
    'left of','to the left of','to the left',
    'right of','to the right of','to the right',
    'above','on top of','over','on',
    'below','under','beneath','underneath',
    'in front of','behind','in back of',
    'inside','outside',
}

def _classify_rel(rel):
    r = rel.lower().strip()
    if r in _SPATIAL_RELS: return 'SPATIAL'
    _ACTION_WORDS = {'riding','holding','carrying','eating','drinking','wearing',
                     'pushing','pulling','walking','running','sitting','standing',
                     'playing','using','throwing','catching','driving','feeding',
                     'reading','watching','kicking','touching','hugging','kissing',
                     'jumping','climbing','mounted','chained'}
    if any(w in r for w in _ACTION_WORDS): return 'ACTION'
    return 'ATTRIBUTE'

def _clean_label(label):
    label = label.strip().lower()
    words = label.split()
    while words and words[0] in ('a','an','the'): words = words[1:]
    return ' '.join(words) if words else label

def _core_noun(phrase):
    stop = {'a','an','the','of','in','on','at','is','its','some','with',
            'left','right','old','young','small','large','big','little'}
    words = [w for w in phrase.lower().split() if w not in stop and len(w) >= 2]
    return words[-1] if words else phrase.lower().strip()

def detect_objects(image, queries):
    text = '. '.join(queries) + '.'
    inputs = gdino_processor(images=image, text=text, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = gdino_model(**inputs)
    results = gdino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=DETECTION_THRESHOLD, text_threshold=DETECTION_THRESHOLD,
        target_sizes=[image.size[::-1]])[0]
    W, H = image.size
    lkey = 'text_labels' if 'text_labels' in results else 'labels'
    dets = []
    for score, label, box in zip(results['scores'], results[lkey], results['boxes']):
        x1, y1, x2, y2 = box.tolist()
        dets.append((_clean_label(label), score.item(), [x1/W, y1/H, x2/W, y2/H]))
    return dets

def _find_best_bbox(entity, detections):
    core = _core_noun(entity)
    best_score, best_box = -1, None
    for label, score, box in detections:
        label_core = _core_noun(label)
        if core == label_core or core in label_core or label_core in core:
            if score > best_score: best_score, best_box = score, box
    return best_box

def _crop_to_bboxes(pil_image, box1, box2, padding=0.15):
    W, H = pil_image.size
    xs = [box1[0], box1[2], box2[0], box2[2]]
    ys = [box1[1], box1[3], box2[1], box2[3]]
    x1 = max(0.0, min(xs) - padding); y1 = max(0.0, min(ys) - padding)
    x2 = min(1.0, max(xs) + padding); y2 = min(1.0, max(ys) + padding)
    left, top, right, bottom = int(x1*W), int(y1*H), int(x2*W), int(y2*H)
    if right - left < 32 or bottom - top < 32: return pil_image
    return pil_image.crop((left, top, right, bottom))

def _spatial_verdict_from_boxes(subj_box, obj_box, rel):
    cx_s = (subj_box[0] + subj_box[2]) / 2
    cy_s = (subj_box[1] + subj_box[3]) / 2
    cx_o = (obj_box[0] + obj_box[2]) / 2
    cy_o = (obj_box[1] + obj_box[3]) / 2
    r = rel.lower().strip()
    THRESH = 0.08
    if r in ('left of','to the left of','to the left'):
        if cx_s < cx_o - THRESH: return True
        if cx_s > cx_o + THRESH: return False
        return None
    if r in ('right of','to the right of','to the right'):
        if cx_s > cx_o + THRESH: return True
        if cx_s < cx_o - THRESH: return False
        return None
    if r in ('above','over'):
        if cy_s < cy_o - THRESH: return True
        if cy_s > cy_o + THRESH: return False
        return None
    if r in ('below','under','beneath','underneath'):
        if cy_s > cy_o + THRESH: return True
        if cy_s < cy_o - THRESH: return False
        return None
    if r in ('on top of','on'):
        horiz_overlap = subj_box[0] < obj_box[2] and subj_box[2] > obj_box[0]
        if cy_s < cy_o - THRESH and horiz_overlap: return True
        if cy_s > cy_o + THRESH: return False
        return None
    if r in ('in front of',):
        if cy_s > cy_o + THRESH: return True
        if cy_s < cy_o - THRESH: return False
        return None
    if r in ('behind','in back of'):
        if cy_s < cy_o - THRESH: return True
        if cy_s > cy_o + THRESH: return False
        return None
    return None

_COUNTERFACTUAL_MAP = {
    'riding':'standing next to','sitting on':'standing near',
    'holding':'standing next to','carrying':'walking away from',
    'wearing':'next to','eating':'looking at','pulling':'pushing',
    'pushing':'pulling','throwing':'holding','catching':'dropping',
    'driving':'standing near','leading':'following',
    'playing with':'ignoring','using':'near',
    'standing on':'next to','lying on':'sitting near',
    'hanging from':'standing near','leaning on':'standing near',
}

def _verify_triple(subj, rel, obj_, detections, pil_image):
    rel_type = _classify_rel(rel)
    def _get_b64(pil):
        buf = BytesIO()
        pil.convert('RGB').save(buf, format='JPEG', quality=85)
        return base64.b64encode(buf.getvalue()).decode()
    if rel_type == 'SPATIAL' and detections:
        subj_box = _find_best_bbox(subj, detections)
        obj_box  = _find_best_bbox(obj_, detections)
        if subj_box and obj_box:
            verdict = _spatial_verdict_from_boxes(subj_box, obj_box, rel)
            if verdict is not None: return verdict
    if pil_image is None: return None
    subj_box = _find_best_bbox(subj, detections) if detections else None
    obj_box  = _find_best_bbox(obj_, detections) if detections else None
    using_crop = bool(subj_box and obj_box)
    crop = _crop_to_bboxes(pil_image, subj_box, obj_box, padding=0.15) if using_crop else pil_image
    region_hint = 'this cropped region' if using_crop else 'the full image'
    crop_b64 = _get_b64(crop)
    yes_v = no_v = 0
    for q in [
        f'In this image, is the {subj} {rel} the {obj_}? Look carefully at {region_hint}. Answer YES or NO only.',
        f'Does the {subj} appear to be {rel} the {obj_} here? Observe {region_hint} closely. Answer YES or NO only.',
    ]:
        r = vlm_call([{'role':'user','content':[
            {'type':'image_url','image_url':{'url':f'data:image/jpeg;base64,{crop_b64}'}},
            {'type':'text','text':q}]}], max_tokens=5)
        if not r: continue
        rl = r.strip().lower()
        if 'yes' in rl and 'no' not in rl: yes_v += 1
        elif 'no' in rl: no_v += 1
    cf_rel = _COUNTERFACTUAL_MAP.get(rel.lower(), f'not {rel}')
    ab_flip = (hash(f'{subj}{rel}{obj_}') % 2 == 1)
    opt_a, opt_b = (cf_rel, rel) if ab_flip else (rel, cf_rel)
    cf_q = (f'Look at {region_hint} carefully. Which is more accurate?\n'
            f'(A) The {subj} is {opt_a} the {obj_}\n'
            f'(B) The {subj} is {opt_b} the {obj_}\n'
            f'Answer ONLY the letter A or B.')
    cf_r = vlm_call([{'role':'user','content':[
        {'type':'image_url','image_url':{'url':f'data:image/jpeg;base64,{crop_b64}'}},
        {'type':'text','text':cf_q}]}], max_tokens=5)
    if cf_r:
        cr = cf_r.strip().upper()
        chose_a = ('A' in cr and 'B' not in cr)
        chose_b = ('B' in cr and 'A' not in cr)
        if ab_flip:
            if chose_b: yes_v += 1
            elif chose_a: no_v += 1
        else:
            if chose_a: yes_v += 1
            elif chose_b: no_v += 1
    total = yes_v + no_v
    if total == 0: return None
    if yes_v > no_v: return True
    if no_v > yes_v: return False
    return None

print('RelCheck verifier ready.')

In [ ]:
# ============================================================
# Cell 2 — Load R-Bench Data + Images
# ============================================================
import pathlib

os.makedirs(SAVE_DIR, exist_ok=True)

# ── Download R-Bench if not cached ──
if not os.path.exists(RBENCH_PATH):
    print('R-Bench data not found. Downloading...')
    RBENCH_DIR = '/content/R-Bench'
    if not os.path.exists(RBENCH_DIR):
        !git clone https://github.com/mrwu-mac/R-Bench {RBENCH_DIR}
    DRIVE_FILE_ID = '1sqO0MWBg_HXp5cIKb-nstjNEEk5crUWH'
    ANNOTATIONS_ZIP = f'{RBENCH_DIR}/rbench_annotations.zip'
    !gdown --id {DRIVE_FILE_ID} -O {ANNOTATIONS_ZIP} -q
    import zipfile
    with zipfile.ZipFile(ANNOTATIONS_ZIP, 'r') as z:
        z.extractall(RBENCH_DIR)
    all_jsons = sorted(pathlib.Path(RBENCH_DIR).rglob('*.json'))
    rbench_raw = None
    for f in all_jsons:
        if 'image-level' in f.name.lower() or 'image_level' in f.name.lower():
            with open(f) as fh: data = json.load(fh)
            if isinstance(data, list) and len(data) > 50:
                rbench_raw = data; break
    if rbench_raw is None:
        for f in all_jsons:
            with open(f) as fh: data = json.load(fh)
            if isinstance(data, list) and len(data) > 100:
                rbench_raw = data; break
    rbench_data = {}
    for entry in rbench_raw:
        img = entry.get('image', entry.get('img', ''))
        img_id = os.path.splitext(os.path.basename(img))[0]
        q = str(entry.get('text', entry.get('question', '')))
        a = str(entry.get('label', entry.get('answer', ''))).lower().strip()
        if img_id and q:
            rbench_data.setdefault(img_id, []).append({'question': q, 'answer': a})
    with open(RBENCH_PATH, 'w') as f: json.dump(rbench_data, f)
    print(f'Saved {len(rbench_data)} images, {sum(len(v) for v in rbench_data.values())} questions')
else:
    with open(RBENCH_PATH) as f: rbench_data = json.load(f)
    print(f'Loaded R-Bench: {len(rbench_data)} images, {sum(len(v) for v in rbench_data.values())} questions')

# ── Load images ──
if not os.path.exists(DRIVE_IMAGES_DIR):
    print(f'Images dir not found: {DRIVE_IMAGES_DIR}')
    print('Please download nocaps images and place in that folder.')
else:
    all_files = list(pathlib.Path(DRIVE_IMAGES_DIR).glob('*.jpg'))
    available = {f.stem: str(f) for f in all_files}
    pool = [(img_id, qs) for img_id, qs in rbench_data.items() if img_id in available]
    random.seed(RANDOM_SEED)
    selected = random.sample(pool, min(N_IMAGES, len(pool)))
    
    pil_images = {}
    rbench_questions = {}
    for img_id, qs in selected:
        try:
            pil_images[img_id] = Image.open(available[img_id]).convert('RGB')
            rbench_questions[img_id] = qs
        except Exception as e:
            print(f'  [{img_id}] load error: {e}')
    
    print(f'Loaded {len(pil_images)} images with {sum(len(v) for v in rbench_questions.values())} R-Bench questions')


In [ ]:
# ============================================================
# Cell 3 — Caption Generation (incremental)
# ============================================================
CAPTIONS_PATH = f'{SAVE_DIR}/{CAPTIONER}_captions.json'

DESCRIBE_PROMPT = (
    'Describe this image in detail. Include all objects, '
    'their spatial positions relative to each other, any actions '
    'or interactions taking place, and notable attributes like colors and sizes.'
)

def caption_image_api(pil_image, model, max_tokens=300):
    buf = BytesIO()
    pil_image.convert('RGB').save(buf, format='JPEG', quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode()
    return llm_call(
        [{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{b64}'}},
            {'type': 'text', 'text': DESCRIBE_PROMPT}]}],
        model=model, max_tokens=max_tokens)

def caption_image_llava(pil_image, max_new_tokens=300):
    if llava_model_local is None: return None
    conversation = [{'role': 'user', 'content': [
        {'type': 'image'},
        {'type': 'text', 'text': DESCRIBE_PROMPT}]}]
    prompt_text = llava_processor_local.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = llava_processor_local(images=pil_image, text=prompt_text, return_tensors='pt').to(llava_model_local.device)
    with torch.no_grad():
        out = llava_model_local.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen_ids = out[:, inputs['input_ids'].shape[1]:]
    return llava_processor_local.decode(gen_ids[0], skip_special_tokens=True).strip()

def caption_image_blip2(pil_image, max_new_tokens=50):
    if blip2_model_local is None: return None
    inputs = blip2_processor_local(images=pil_image, return_tensors='pt').to(blip2_model_local.device, torch.float16)
    with torch.no_grad():
        out = blip2_model_local.generate(**inputs, max_new_tokens=max_new_tokens)
    return blip2_processor_local.decode(out[0], skip_special_tokens=True).strip()

def caption_image(pil_image):
    if CAPTIONER == 'llava' and llava_model_local is not None:
        return caption_image_llava(pil_image)
    elif CAPTIONER == 'blip2' and blip2_model_local is not None:
        return caption_image_blip2(pil_image)
    elif CAPTIONER == 'blip2':
        print('  WARNING: BLIP-2 not loaded locally. Skipping.')
        return None
    else:
        return caption_image_api(pil_image, _CAPTIONER_MODELS.get(CAPTIONER, VLM_MODEL))

# Load cached + generate missing
if Path(CAPTIONS_PATH).exists():
    with open(CAPTIONS_PATH) as f: captions = json.load(f)
    print(f'Loaded {len(captions)} cached {CAPTIONER} captions')
else:
    captions = {}

missing = [img_id for img_id in pil_images if img_id not in captions]
if missing:
    print(f'Generating {len(missing)} new captions with {CAPTIONER}...')
    for img_id in missing:
        cap = caption_image(pil_images[img_id])
        if cap:
            captions[img_id] = cap
            print(f'  [{img_id}] {len(cap.split())}w: {cap[:80]}...')
    with open(CAPTIONS_PATH, 'w') as f: json.dump(captions, f)

avg_words = sum(len(c.split()) for c in captions.values()) / max(len(captions), 1)
print(f'\nCaptioned {len(captions)} images (avg {avg_words:.0f} words)')

In [ ]:
# ============================================================
# Cell 4 — Extract Triples + Inject Hallucinations
# ============================================================
# For each caption:
#   1. Extract relational triples via Llama
#   2. For each triple, if the relation has a known contradiction,
#      inject it → create a "corrupted" caption
#   3. Save: original caption, corrupted caption, injected hallucination

TRIPLE_PROMPT_TMPL = ('Extract relational claims from this caption as JSON.\n'
                      'Caption: "CAPTION_PLACEHOLDER"\n'
                      'Output a JSON array: [{"subject": "...", "relation": "...", "object": "..."}]\n'
                      'Only claims about how two entities relate. ONLY valid JSON, no explanation.')

# ── Injection table: original relation → hallucinated replacement ──
# These are REVERSE of the TRAP_TABLE: given a correct relation, what
# would a hallucinating captioner say instead?
INJECTION_TABLE = {
    # Action injections (replace correct with stereotypical hallucination)
    'sitting on':       'riding',
    'standing next to': 'riding',
    'standing near':    'holding',
    'next to':          'riding',
    'near':             'holding',
    'looking at':       'eating',
    'walking near':     'carrying',

    # Spatial injections (swap with opposite)
    'left of':          'right of',
    'right of':         'left of',
    'to the left of':   'to the right of',
    'to the right of':  'to the left of',
    'above':            'below',
    'below':            'above',
    'on top of':        'under',
    'under':            'on top of',
    'on':               'under',
    'in front of':      'behind',
    'behind':           'in front of',
}

INJECT_PATH = f'{SAVE_DIR}/injected_{CAPTIONER}.json'

if Path(INJECT_PATH).exists():
    with open(INJECT_PATH) as f: injected_data = json.load(f)
    print(f'Loaded {len(injected_data)} cached injection records')
else:
    injected_data = {}
    print('Extracting triples and injecting hallucinations...')

    for img_id, caption in captions.items():
        # Use .replace() to avoid KeyError if caption contains curly braces
        prompt = TRIPLE_PROMPT_TMPL.replace('CAPTION_PLACEHOLDER', caption)
        raw = llm_call([{'role': 'user', 'content': prompt}], max_tokens=400)
        try:
            clean = re.sub(r'```[a-z]*\n?', '', raw or '').replace('```', '').strip()
            triples = json.loads(clean)
        except Exception:
            triples = []

        # Find injectable triples
        injections = []
        for ct in triples:
            s = ct.get('subject', '').strip()
            r = ct.get('relation', '').lower().strip()
            o = ct.get('object', '').strip()
            if not s or not r or not o: continue

            halluc_rel = INJECTION_TABLE.get(r)
            if halluc_rel is None: continue

            # Create corrupted caption: replace the relation in the text
            corrupted = caption.replace(r, halluc_rel, 1)
            if corrupted == caption:
                # Try case-insensitive
                corrupted = re.sub(re.escape(r), halluc_rel, caption, count=1, flags=re.IGNORECASE)
            if corrupted == caption: continue  # couldn't inject

            injections.append({
                'subject': s, 'original_rel': r, 'halluc_rel': halluc_rel, 'object': o,
                'original_caption': caption, 'corrupted_caption': corrupted,
                'triple_str': f'({s}, {halluc_rel}, {o})',
            })

        if injections:
            # Keep only the first injection per image (simplest test)
            injected_data[img_id] = injections[0]
            inj = injections[0]
            print(f'  [{img_id}] ({inj["subject"]}, {inj["original_rel"]} -> {inj["halluc_rel"]}, {inj["object"]})')

    with open(INJECT_PATH, 'w') as f: json.dump(injected_data, f, indent=2)

n_inj = len(injected_data)
n_total = len(captions)
print(f'\nInjected hallucinations: {n_inj}/{n_total} images ({100*n_inj/max(n_total,1):.0f}%)')
if n_inj == 0:
    print('WARNING: No injectable triples found. Captions may not contain relations in INJECTION_TABLE.')

In [ ]:
# ============================================================
# Cell 5 — RelCheck Detection on Corrupted Captions
# ============================================================
# For each injected hallucination:
#   Run _verify_triple() on the CORRUPTED triple
#   verdict=False → DETECTED (correct!)
#   verdict=True  → MISSED  (false negative)
#   verdict=None  → UNCERTAIN

DETECT_PATH = f'{SAVE_DIR}/detection_results_{CAPTIONER}.json'

print(f'Running RelCheck detection on {len(injected_data)} corrupted triples...\n')

detection_results = []
detected = missed = uncertain = 0

for img_id, inj in injected_data.items():
    pil = pil_images.get(img_id)
    if pil is None:
        print(f'  [{img_id}] image not loaded — skipping')
        continue

    s, r, o = inj['subject'], inj['halluc_rel'], inj['object']
    rel_type = _classify_rel(r)

    # Run GDino detection
    all_entities = list(set([s, o]))
    detections = detect_objects(pil, all_entities)

    # Run RelCheck verifier
    verdict = _verify_triple(s, r, o, detections, pil)

    if verdict is False:
        status = 'DETECTED'
        detected += 1
    elif verdict is True:
        status = 'MISSED'
        missed += 1
    else:
        status = 'UNCERTAIN'
        uncertain += 1

    result = {
        'img_id': img_id,
        'subject': s, 'halluc_rel': r, 'object': o,
        'original_rel': inj['original_rel'],
        'rel_type': rel_type,
        'verdict': str(verdict),
        'status': status,
    }
    detection_results.append(result)
    icon = {'DETECTED': '\u2713', 'MISSED': '\u2717', 'UNCERTAIN': '?'}[status]
    print(f'  [{img_id}] {icon} {status}: ({s}, {r}, {o}) [{rel_type}]')

with open(DETECT_PATH, 'w') as f:
    json.dump(detection_results, f, indent=2)

total = detected + missed + uncertain
print(f'\n\u2500\u2500 Detection Results \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'  Total injected:  {total}')
print(f'  DETECTED:        {detected} ({100*detected/max(total,1):.1f}%)')
print(f'  MISSED:          {missed} ({100*missed/max(total,1):.1f}%)')
print(f'  UNCERTAIN:       {uncertain} ({100*uncertain/max(total,1):.1f}%)')
print(f'  Detection recall: {100*detected/max(total,1):.1f}%')

# Breakdown by relation type
by_type = defaultdict(lambda: {'detected':0, 'missed':0, 'uncertain':0})
for r in detection_results:
    by_type[r['rel_type']][r['status'].lower()] += 1
print(f'\n  By relation type:')
for rtype, counts in sorted(by_type.items()):
    t = sum(counts.values())
    d = counts['detected']
    print(f'    {rtype:12} — {d}/{t} detected ({100*d/max(t,1):.0f}%)')


In [ ]:
# ============================================================
# Cell 6 — R-POPE LLM-Judge Evaluation
# ============================================================
# Compare R-POPE accuracy:
#   A) Original caption (before injection)
#   B) Corrupted caption (with injected hallucination)

RPOPE_PROMPT_TMPL = ('Based ONLY on this description, answer the question with Yes or No.\n\n'
                     'Description: "CAPTION_PLACEHOLDER"\n'
                     'Question: QUESTION_PLACEHOLDER\n\n'
                     'Answer ONLY Yes or No.')

def rpope_judge(caption, question):
    # Use .replace() to avoid KeyError if caption/question contain curly braces
    prompt = RPOPE_PROMPT_TMPL.replace('CAPTION_PLACEHOLDER', caption).replace('QUESTION_PLACEHOLDER', question)
    resp = llm_call([{'role': 'user', 'content': prompt}], max_tokens=5)
    if not resp: return None
    r = resp.strip().lower()
    if 'yes' in r and 'no' not in r: return 'yes'
    if 'no' in r: return 'no'
    return None

print('Running R-POPE LLM-Judge evaluation...\n')

# Only evaluate images that have both injections AND R-Bench questions
eval_images = [img_id for img_id in injected_data if img_id in rbench_questions]
print(f'Evaluating {len(eval_images)} images with R-Bench questions')

correct_original = correct_corrupted = total_qs = 0

for img_id in eval_images:
    inj = injected_data[img_id]
    qs = rbench_questions.get(img_id, [])

    for qa in qs:
        question = qa['question']
        gt = qa['answer'].lower().strip()
        if gt not in ('yes', 'no'): continue

        # Judge original caption
        orig_ans = rpope_judge(inj['original_caption'], question)
        # Judge corrupted caption
        corr_ans = rpope_judge(inj['corrupted_caption'], question)

        if orig_ans == gt: correct_original += 1
        if corr_ans == gt: correct_corrupted += 1
        total_qs += 1

if total_qs > 0:
    acc_orig = 100 * correct_original / total_qs
    acc_corr = 100 * correct_corrupted / total_qs
    delta = acc_orig - acc_corr
    print(f'\n── R-POPE Results ────────────────────────────────────────')
    print(f'  Questions evaluated: {total_qs}')
    print(f'  Original accuracy:  {acc_orig:.1f}% ({correct_original}/{total_qs})')
    print(f'  Corrupted accuracy: {acc_corr:.1f}% ({correct_corrupted}/{total_qs})')
    print(f'  Accuracy DROP:      {delta:+.1f}% (injection impact)')
    print()
    print(f'  If RelCheck detects + corrects the injection,')
    print(f'  accuracy should recover from {acc_corr:.1f}% back toward {acc_orig:.1f}%.')
else:
    print('WARNING: No R-Bench questions matched evaluated images.')